# Exercícios Aula 04: Detecção de Anomalias

Este notebook contém **5 exercícios práticos** para reforçar os principais conceitos apresentados nos notebooks da Aula 04 (`04_01` e `04_02`):

1. Carregando o modelo e interpretando uma previsão individual
2. Avaliando um lote de imagens
3. Visualizando heatmap e máscara em diferentes defeitos
4. Gerando um laudo em texto livre com VLM
5. Laudo estruturado (JSON) e sinalização para revisão humana (HITL)

Complete os blocos marcados com `# TODO` em cada exercício. Use os notebooks `04_01` e `04_02` como referência sempre que precisar relembrar a sintaxe de alguma função.

**Observação:** os Exercícios 4 e 5 exigem uma chave de API do OpenRouter. Consulte o [README.md](README.md) do repositório para saber como obtê-la.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install anomalib==2.5.1
    !pip install openai
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Vamos combinar as bibliotecas usadas em `04_01_deteccao_anomalias.ipynb` e em `04_02_anomalia_vlm.ipynb`: `anomalib` (modelo especializado) e `openai` (VLM via OpenRouter).

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from getpass import getpass
from pathlib import Path
import json
import base64
import mimetypes

from anomalib.data import PredictDataset
from anomalib.models import EfficientAd
from anomalib.engine import Engine

from openai import OpenAI

import torch

import numpy as np
import matplotlib.pyplot as plt
# Indica ao notebook to render figures in-page.
%matplotlib inline
from IPython.display import Markdown

## Preparação: Modelo de Detecção de Anomalias

Antes dos Exercícios 1 a 3, vamos carregar o `EfficientAd` a partir do checkpoint treinado (`modelos/efficientad-bottle-epoch=19.ckpt`) e reaproveitar a função `apresentar_previsao()` do notebook `04_01`. Essa célula é apenas infraestrutura, não faz parte dos exercícios.

In [ ]:
model = EfficientAd()

engine = Engine(
    accelerator="cpu",
    devices=1,
    enable_progress_bar=False,
    logger=False,
    default_root_dir="output/results"
)

checkpoint_path = "modelos/efficientad-bottle-epoch=19.ckpt"

def apresentar_previsao(item) -> None:

    status = "ANOMALIA" if item.pred_label else "NORMAL"

    # ------------------
    # Imagem
    # ------------------
    img = item.image

    if torch.is_tensor(img):
        img = img.detach().cpu().permute(1, 2, 0).numpy()

    img = np.clip(img, 0.0, 1.0)

    # ------------------
    # Anomaly Map
    # ------------------
    anomaly_map = item.anomaly_map

    if torch.is_tensor(anomaly_map):
        anomaly_map = anomaly_map.detach().cpu().numpy()

    anomaly_map_norm = (anomaly_map - anomaly_map.min()) / (anomaly_map.max() - anomaly_map.min() + 1e-8)

    # ------------------
    # Máscara do objeto
    # ------------------
    gray = img.mean(axis=2)

    # Fundo branco
    obj_mask = gray < 0.95

    # Aplicar máscara
    anomaly_map_masked = anomaly_map_norm.copy()
    anomaly_map_masked[~obj_mask] = np.nan

    # ------------------
    # Destacar apenas regiões mais relevantes
    # ------------------
    threshold = np.percentile(
        anomaly_map_masked[~np.isnan(anomaly_map_masked)],
        90
    )

    overlay_map = anomaly_map_masked.copy()
    overlay_map[overlay_map < threshold] = np.nan

    # ------------------
    # Pred Mask
    # ------------------
    pred_mask = item.pred_mask

    if torch.is_tensor(pred_mask):
        pred_mask = pred_mask.detach().cpu().numpy()

    # ------------------
    # Plot
    # ------------------
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img)
    axes[0].set_title("Imagem")
    axes[0].axis("off")

    axes[1].imshow(anomaly_map_masked, cmap="jet")
    axes[1].set_title("Mapa de Calor")
    axes[1].axis("off")

    axes[2].imshow(img)

    if pred_mask is not None:
        axes[2].contour(
            pred_mask,
            levels=[0.5],
            colors="red",
            linewidths=2
        )

    axes[2].set_title("Máscara")
    axes[2].axis("off")

    plt.suptitle(
        f"{Path(item.image_path).name} | {status} | score={float(item.pred_score):.3f}",
        fontsize=18,
        fontweight="bold"
    )

    plt.show()

print("Modelo e função de apresentação prontos!")

## Exercício 1: Carregando o Modelo e Interpretando uma Previsão Individual

**Conceito reforçado:** um modelo de detecção de anomalias é treinado apenas com imagens normais; na inferência, `pred_score` (limiar de `0.5`) e `pred_label` indicam se a imagem foi considerada anômala, e `anomaly_map` traz o heatmap pixel a pixel dessa pontuação.

1. Rode `engine.predict()` na imagem `imagens/04/bottle_test/contamination/000.png` usando `PredictDataset`.
2. Pegue a primeira previsão da lista retornada.
3. Imprima o `pred_score` e o `pred_label` dessa previsão.
4. Normalize e apresente o `anomaly_map` (heatmap) com `plt.imshow(..., cmap='jet')`.

In [ ]:
# 1. Rode engine.predict() na imagem 'imagens/04/bottle_test/contamination/000.png'


In [ ]:
# 2. Pegue a primeira previsão da lista retornada


In [ ]:
# 3. Imprima o pred_score e o pred_label


In [ ]:
# 4. Normalize e apresente o anomaly_map com plt.imshow(cmap='jet')


## Exercício 2: Avaliando um Lote de Imagens

**Conceito reforçado:** `engine.predict()` também aceita o caminho de uma pasta (`data_path`) e roda o modelo em todas as imagens dela; assim, podemos avaliar a taxa de detecção do modelo comparando pastas normais e anômalas.

1. Rode `engine.predict()` na pasta `imagens/04/bottle_test/broken_small`.
2. Desempacote a lista de `ImageBatch` retornada em uma lista simples de itens (igual ao notebook `04_01`).
3. Conte quantos itens dessa lista foram classificados como anomalia (`pred_label=True`).
4. Calcule e imprima a taxa de detecção (percentual de imagens classificadas como anomalia).

In [ ]:
# 1. Rode engine.predict() na pasta 'imagens/04/bottle_test/broken_small'


In [ ]:
# 2. Desempacote a lista de ImageBatch em uma lista simples de itens


In [ ]:
# 3. Conte quantos itens foram classificados como anomalia (pred_label=True)


In [ ]:
# 4. Calcule e imprima a taxa de detecção (%)


## Exercício 3: Visualizando Heatmap e Máscara em Diferentes Defeitos

**Conceito reforçado:** `anomaly_map` traz a intensidade da anomalia pixel a pixel, enquanto `pred_mask` traz a região binária que o modelo aponta como defeito. Diferentes tipos de defeito tendem a acionar regiões distintas do objeto.

1. Rode a previsão em uma imagem de `imagens/04/bottle_test/broken_large/000.png`.
2. Rode a previsão em uma imagem de `imagens/04/bottle_test/contamination/000.png`.
3. Apresente as duas previsões usando `apresentar_previsao()`.
4. Compare visualmente: o heatmap e a máscara aparecem em regiões diferentes da garrafa? Escreva sua conclusão em um comentário.

In [ ]:
# 1. Rode a previsão em 'imagens/04/bottle_test/broken_large/000.png'


In [ ]:
# 2. Rode a previsão em 'imagens/04/bottle_test/contamination/000.png'


In [ ]:
# 3. Apresente as duas previsões usando apresentar_previsao()


In [ ]:
# 4. Compare as regiões destacadas e escreva sua conclusão:


## Preparação: VLM e Geração de Evidências

Antes dos Exercícios 4 e 5, vamos configurar o cliente do OpenRouter e reaproveitar as funções `image_to_data_url()`, `perguntar_imagens()` e `gerar_evidencias()` do notebook `04_02`. Você vai precisar de uma API Key do OpenRouter (veja o [README.md](README.md)).

In [ ]:
API_KEY = getpass("Digite a API KEY do OpenRouter")
OPEN_ROUTER_DEFAULT_MODEL = 'openrouter/free'

client = OpenAI(
    api_key=API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

def image_to_data_url(image_path: str) -> str:
    """Converte uma imagem para Data URL compatível com OpenAI/OpenRouter."""
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {image_path}")

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None and str(image_path).endswith('.webp'):
        mime_type = "image/webp"

    if mime_type not in {"image/jpeg", "image/png", "image/webp"}:
        raise ValueError(f"Formato não suportado: {mime_type}")

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"

def perguntar_imagens(image_paths: list, prompt: str, model: str = None):
    content = [{"type": "text", "text": prompt}]

    for image_path in image_paths:
        content.append({
            "type": "image_url",
            "image_url": {"url": image_to_data_url(image_path)}
        })

    response = client.chat.completions.create(
        model=model or OPEN_ROUTER_DEFAULT_MODEL,
        messages=[{"role": "user", "content": content}]
    )

    return response.choices[0].message.content

def gerar_evidencias(item, saida_dir: str = "output/evidencias") -> dict:
    saida_dir = Path(saida_dir)
    saida_dir.mkdir(parents=True, exist_ok=True)

    img = item.image
    if torch.is_tensor(img):
        img = img.detach().cpu().permute(1, 2, 0).numpy()
    img = np.clip(img, 0.0, 1.0)

    anomaly_map = item.anomaly_map
    if torch.is_tensor(anomaly_map):
        anomaly_map = anomaly_map.detach().cpu().numpy()
    anomaly_map = anomaly_map.squeeze()
    anomaly_map_norm = (anomaly_map - anomaly_map.min()) / (anomaly_map.max() - anomaly_map.min() + 1e-8)

    gray = img.mean(axis=2)
    obj_mask = gray < 0.95

    anomaly_map_masked = anomaly_map_norm.copy()
    anomaly_map_masked[~obj_mask] = np.nan

    pred_mask = item.pred_mask
    if torch.is_tensor(pred_mask):
        pred_mask = pred_mask.detach().cpu().numpy()
    pred_mask = pred_mask.squeeze()

    nome = f"{Path(item.image_path).parent.name}_{Path(item.image_path).stem}"
    caminhos = {}

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    ax.axis("off")
    caminho = saida_dir / f"{nome}_original.png"
    fig.savefig(caminho, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    caminhos["original"] = str(caminho)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    ax.imshow(anomaly_map_masked, cmap="jet", alpha=0.6)
    ax.axis("off")
    caminho = saida_dir / f"{nome}_heatmap.png"
    fig.savefig(caminho, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    caminhos["heatmap"] = str(caminho)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    ax.contour(pred_mask, levels=[0.5], colors="red", linewidths=2)
    ax.axis("off")
    caminho = saida_dir / f"{nome}_contorno.png"
    fig.savefig(caminho, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    caminhos["contorno"] = str(caminho)

    status = "ANOMALIA" if item.pred_label else "NORMAL"

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(plt.imread(caminhos["original"])); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(plt.imread(caminhos["heatmap"])); axes[1].set_title("Heatmap"); axes[1].axis("off")
    axes[2].imshow(plt.imread(caminhos["contorno"])); axes[2].set_title("Contorno"); axes[2].axis("off")
    plt.suptitle(
        f"{Path(item.image_path).name} | {status} | score={float(item.pred_score):.3f}",
        fontsize=16,
        fontweight="bold"
    )
    plt.show()

    return caminhos

print("Cliente OpenRouter e funções auxiliares prontos!")

## Exercício 4: Gerando um Laudo em Texto Livre com VLM

**Conceito reforçado:** fornecer ao VLM as evidências do modelo especializado (imagem original, heatmap, contorno e score) em vez de perguntar diretamente "existe uma anomalia?" reduz o risco de alucinação, pois o VLM interpreta uma detecção concreta em vez de "adivinhar" sozinho.

1. Rode a previsão na imagem `imagens/04/bottle_test/broken_small/000.png` e pegue o item da previsão (`predictions[0].items[0]`).
2. Gere as evidências visuais com `gerar_evidencias()`.
3. Monte um prompt próprio pedindo um laudo técnico em texto livre com base nessas evidências (inspire-se em `montar_prompt_laudo()` do notebook `04_02`).
4. Rode `perguntar_imagens()` passando as 3 evidências e o prompt, e apresente o laudo com `Markdown()`.

In [ ]:
# 1. Rode a previsão em 'imagens/04/bottle_test/broken_small/000.png' e pegue o item


In [ ]:
# 2. Gere as evidências visuais com gerar_evidencias()


In [ ]:
# 3. Monte seu próprio prompt pedindo um laudo técnico em texto livre


In [ ]:
# 4. Rode perguntar_imagens() com as evidências e apresente o laudo


## Exercício 5: Laudo Estruturado (JSON) e Sinalização para Revisão Humana (HITL)

**Conceito reforçado:** pedir a saída em JSON (schema) permite armazenar o laudo de forma estruturada, e um campo como `concorda_com_modelo_especializado` pode servir de gatilho automático para revisão humana sempre que o VLM divergir do modelo especializado.

1. Usando a mesma imagem do Exercício 4, defina um schema JSON simples para o laudo, incluindo ao menos os campos `status` e `concorda_com_modelo_especializado`.
2. Monte o prompt pedindo a saída em JSON conforme esse schema (inspire-se em `montar_prompt_laudo_json()` do notebook `04_02`).
3. Rode `perguntar_imagens()` e apresente o resultado.
4. Repita o processo com uma imagem de `imagens/04/bottle_test/good/000.png` (normal). O VLM concordou com o modelo especializado nas duas imagens? Em que situação você acionaria uma revisão humana? Escreva sua conclusão em um comentário.

In [ ]:
# 1. Defina um schema JSON simples com pelo menos status e concorda_com_modelo_especializado


In [ ]:
# 2. Monte o prompt pedindo a saída em JSON conforme esse schema


In [ ]:
# 3. Rode perguntar_imagens() e apresente o resultado


In [ ]:
# 4. Repita com uma imagem 'good', compare e escreva sua conclusão sobre HITL:
